In [1]:
# Standard Libraries
import os
import json
import random
import functools
import copy
from typing import Optional, List

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset, TensorDataset

# Pyro (Probabilistic Programming)
import pyro
from pyro.infer import SVI, TraceMeanField_ELBO, Trace_ELBO
from pyro.nn.module import to_pyro_module_

# TyXe (Bayesian Neural Networks)
import tyxe

# Hugging Face Transformers and PEFT (Parameter-Efficient Fine-Tuning)
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import (
    PeftConfig,
    PeftModel,
    LoraConfig,
    get_peft_model,
)
from peft.tuners.lora import LoraLayer

# Hugging Face Datasets
from datasets import Dataset, load_dataset  # Dataset creation and SuperNI dataset

# Accelerate (Efficient Distributed Training)
from accelerate import init_empty_weights

# Hugging Face Hub
from huggingface_hub import login

# BitsAndBytes (Optional: Quantization Optimization)
import bitsandbytes

# NumPy
import numpy as np


In [2]:
os.chdir('/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/SU_data')
target_file = "task511_reddit_tifu_long_text_summarization.json"

with open(target_file, 'r', encoding='utf-8-sig') as f:
    json_data = json.load(f)
from transformers import AutoTokenizer

# Replace 'path/to/seed-model' with your seed model's identifier or local path
seed_model_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_sa_noEVCL2/fine-tuned-sa-lora_noEVCL2"
tokenizer = AutoTokenizer.from_pretrained(seed_model_path)

instances = json_data['Instances'][0:2500]
instruct1="In this task, you are given a Reddit post as a text. Your task is to generate a short summary for this text. The summary must include a situation which caused humor. The summary should be one or two sentences long. \nReddit Post: "
instruct2="\nSummary: "
input_texts = [str(instruct1+instance['input']+instruct2) for instance in instances]
output_texts = [str(instance['output'][0]) if instance['output'] else "" for instance in instances]

print(input_texts[0])
print(output_texts[0])

# Create Hugging Face Dataset
ds = Dataset.from_dict({'input': input_texts, 'output': output_texts})

# Tokenize the dataset
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["output"],
            truncation=True,
            padding="max_length",
            max_length=512
        )
    model_inputs["labels"] = labels["input_ids"]
    model_inputs["attention_mask"] = model_inputs.get("attention_mask", None)
    return model_inputs

# Apply tokenization and set format
tokenized_datasets = ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])
tokenized_datasets.set_format("torch")

# Split dataset into train and eval
train_size = int(0.8 * len(tokenized_datasets))
train_dataset = tokenized_datasets.select(range(train_size))
eval_dataset = tokenized_datasets.select(range(train_size, len(tokenized_datasets)))

# Create DataLoaders
batch_size = 8  
train_loader_2 = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader_2 = DataLoader(eval_dataset, batch_size=batch_size)

# Define data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In this task, you are given a Reddit post as a text. Your task is to generate a short summary for this text. The summary must include a situation which caused humor. The summary should be one or two sentences long. 
Reddit Post: Text: first time poster here, so please forgive any errors or whatnot. i'll edit it if required.

i suppose i should probably start like everyone else and clarify that this did not happen today, but around six years ago.

for my final a-level chemistry coursework, we all had to choose our own investigations (from a list of experiments provided to us) and produce a report of our findings and so on. i chose to undertake the [belousov-zhabotinsky reaction](https://en.wikipedia.org/wiki/belousov%e2%80%93zhabotinsky_reaction). to cut a long story short, this reaction takes place in dilute sulfuric acid.


one of the variables that i happened to be testing the effect of was the concentration of said sulfuric acid, and for some reason, my school decided to entrust a 1

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

/opt/conda/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [3]:
import json_repair 
os.chdir('/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/')
target_file = "SA_final_refined_sampled.jsonl"

with open(target_file, 'r', encoding='utf-8-sig') as f:
    json_data = json_repair.loads(f.read())

instances = json_data
input_texts = [instance['input'] for instance in json_data]
output_texts = [instance['refined_answer'] for instance in json_data]


ds = Dataset.from_dict({'input': input_texts, 'output': output_texts})
tokenized_datasets = ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])
tokenized_datasets.set_format("torch")
train_size = int(1.0 * len(tokenized_datasets))
synthetic_train_dataset = tokenized_datasets.select(range(train_size))
batch_size = 8  
synthetic_loader_1 = DataLoader(synthetic_train_dataset, batch_size=batch_size, shuffle=True)

Map:   0%|          | 0/198 [00:00<?, ? examples/s]

In [4]:
from torch.utils.data import ConcatDataset, DataLoader

# Combine datasets
if synthetic_loader_1 is not None:
    print('combined dataloader')
    combined_dataset = ConcatDataset([train_loader_2.dataset, synthetic_loader_1.dataset])
    combined_loader = DataLoader(combined_dataset, batch_size=8, shuffle=True)
else:
    print('not combined dataloader')
    combined_loader = train_loader_2

combined dataloader


In [5]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
from torch.utils.data import DataLoader

# Set environment variable to manage memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Directories for saving model and offloading
save_dir = os.path.expanduser("fine_tuned_su_noEVCL2/")
offload_dir = os.path.expanduser("offload_folder/")
os.makedirs(save_dir, exist_ok=True)
os.makedirs(offload_dir, exist_ok=True)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(seed_model_path)  # Use the same tokenizer as your QA model
tokenizer.pad_token = tokenizer.eos_token

# Configure bitsandbytes quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True  # Enable FP32 offloading for CPU
)
# Load the QA fine-tuned model
model = AutoModelForCausalLM.from_pretrained(
    seed_model_path,
    device_map="auto",  # Automatically distribute layers across devices
    offload_folder=offload_dir,  # Specify offloading directory for disk storage
    quantization_config=quantization_config,
)

# Configure LoRA for fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Use the combined loader dataset for training
train_dataset = combined_loader.dataset  # Dataset defined in your earlier code
print(f"Length of combined DataLoader before training: {len(train_dataset)}")

# Training arguments
training_args = TrainingArguments(
    output_dir=save_dir,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=2,
    per_device_train_batch_size=4,  # Batch size of 8
    gradient_accumulation_steps=4,  # Adjust based on memory constraints
    num_train_epochs=3,           # Number of epochs
    learning_rate=2e-4,             # Learning rate
    fp16=torch.cuda.is_available(), # Enable mixed precision for GPUs
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Resume training from checkpoint if available
checkpoint_path = os.path.join(save_dir, "checkpoint-latest")
if os.path.exists(checkpoint_path):
    print(f"Resuming training from checkpoint: {checkpoint_path}")
    trainer.train(resume_from_checkpoint=checkpoint_path)
else:
    trainer.train()

# Save the fine-tuned model and tokenizer
model.save_pretrained(os.path.join(save_dir, "fine-tuned-su-lora_noEVCL2"))
tokenizer.save_pretrained(os.path.join(save_dir, "fine-tuned-su-lora_noEVCL2"))

print("Fine-tuning complete. Model and tokenizer saved!")


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Length of combined DataLoader before training: 2198


/tmp/ipykernel_2970/31452509.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


Fine-tuning complete. Model and tokenizer saved!


In [ ]:
!pip install json-repair
